# 🪙 Neiro-Quant — Market Microstructure & GEX Terminal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaniNguyen2509/Neiro-Quant/blob/main/notebooks/Neiro_Quant.ipynb)

> **Hệ thống Định lượng Vị thế Nhà tạo lập (Dealer Positioning), Gamma Exposure (GEX) & Tích hợp Trí tuệ Nhân tạo DeepSeek.**  
> **Tác chiến trước 3 phiên:** Phiên Á (06:30) • Phiên Âu (13:30) • Phiên Mỹ (19:00 VN)  
> **Tài sản:** Gold Futures (`GC` CME) & Gold Spot (`XAUUSD`)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# 📥 THIẾT LẬP ĐƯỜNG DẪN DỮ LIỆU & THƯ MỤC LÀM VIỆC
# ==============================================================================
import os
if os.path.exists('Neiro-Quant'):
    os.chdir('Neiro-Quant')
elif os.path.exists('/content/drive/MyDrive/Neiro-Quant'):
    os.chdir('/content/drive/MyDrive/Neiro-Quant')
print(f'✅ Thư mục làm việc hiện tại: {os.getcwd()}')


### ⚙️ Bước 1: Cài đặt thư viện cần thiết

In [ ]:
# Cài đặt các thư viện định lượng & đồ họa tương tác (Không cần cài SDK AI)
!pip install -q yfinance plotly pandas numpy


### ⚙️ Bước 2: Thiết lập Thông số Phiên Giao dịch (100% Thuần Toán - Không cần API Key)
Nhập giá hiện tại của **CME Gold Futures** và **XAUUSD Spot** trên sàn MT5 của bạn để tự động tính độ lệch Basis.


In [ ]:
# ==============================================================================\n# THÔNG SỐ PHIÊN GIAO DỊCH (Tự động nạp từ Dữ liệu CME Options mới nhất)\n# ==============================================================================\nSESSION_NAME  = 'PHIÊN TÁC CHIẾN GOLD OPTIONS HIỆN TẠI'\nFUTURES_PRICE = 4181.10   # Giá GCZ26 trên CME Futures\nSPOT_PRICE    = 4181.10   # Giá XAUUSD Spot trên Broker MT5\nBASIS         = round(FUTURES_PRICE - SPOT_PRICE, 2)\n\nprint('=' * 65)\nprint(f'📊 THÔNG SỐ ĐẦU VÀO PHIÊN: {SESSION_NAME}')\nprint(f'• Giá Hợp đồng Tương lai (CME Futures): {FUTURES_PRICE:.2f}')\nprint(f'• Giá Vàng Giao ngay (XAUUSD Spot MT5): {SPOT_PRICE:.2f}')\nprint(f'• Độ lệch Basis (Futures - Spot):       {BASIS:+.2f} USD')\nprint('=' * 65)\n

### 📥 Bước 3: Nạp dữ liệu Options Chain
- **Mặc định:** Sử dụng dữ liệu mẫu chuẩn của sàn CME trong kho repo `Neiro-Quant`.
- **Khi tác chiến:** Anh có thể tải file CSV tải từ Barchart lên hoặc dán thẳng vào ô bên dưới.

In [ ]:
# ==============================================================================\n# 📥 BƯỚC 3: BỘ NẠP DỮ LIỆU TỰ ĐỘNG TỪ THƯ MỤC DATA / GOOGLE DRIVE\n# Tự động phát hiện và kết hợp options_prices.csv + volatility-greeks.csv\n# ==============================================================================\nimport pandas as pd\nimport numpy as np\nimport os\nimport re\n\ndef clean_v(v):\n    if v is None or v == 'N/A' or v == '': return 0.0\n    v = str(v).replace('s', '').replace('%', '').replace(',', '').replace('"', '').strip()\n    try: return float(v)\n    except: return 0.0\n\ndef load_and_parse_options_data():\n    # Tìm thư mục chứa file data\n    possible_dirs = [\n        'data', '../data', 'Neiro-Quant/data',\n        '/content/drive/MyDrive/Neiro-Quant/data',\n        '/content/drive/MyDrive/data',\n        'C:/Users/Daniel/Documents/My-Projects/Neiro-Quant/data'\n    ]\n    \n    target_dir = None\n    for d in possible_dirs:\n        if os.path.exists(os.path.join(d, 'options_prices.csv')):\n            target_dir = d\n            break\n            \n    if target_dir is None:\n        # Quét đệ quy tìm kiếm trong Google Drive nếu có\n        if os.path.exists('/content/drive/MyDrive'):\n            for root, dirs, files in os.walk('/content/drive/MyDrive'):\n                if 'options_prices.csv' in files:\n                    target_dir = root\n                    break\n                    \n    if target_dir:\n        print(f'✅ Tìm thấy dữ liệu tại: {target_dir}')\n        prices_file = os.path.join(target_dir, 'options_prices.csv')\n        greeks_file = os.path.join(target_dir, 'volatility-greeks.csv')\n        \n        # 1. Parse Greeks\n        greeks_map = {}\n        if os.path.exists(greeks_file):\n            with open(greeks_file, 'r', encoding='utf-8-sig') as f:\n                g_lines = [l.strip() for l in f if l.strip()]\n            for i in range(266, len(g_lines), 10):\n                if i + 5 < len(g_lines):\n                    try:\n                        k = float(g_lines[i].replace('"', '').replace(',', ''))\n                        iv = clean_v(g_lines[i+3]) / 100.0\n                        gamma = clean_v(g_lines[i+5])\n                        greeks_map[k] = {'iv': iv, 'gamma': gamma}\n                    except: pass\n            print(f'• Đã nạp đường cong Greeks CME ({len(greeks_map)} mốc strike)')\n            \n        # 2. Parse options_prices.csv\n        with open(prices_file, 'r', encoding='utf-8-sig') as f:\n            p_lines = [l.strip() for l in f if l.strip()]\n            \n        calls = {}\n        puts = {}\n        for i in range(14, min(254, len(p_lines)), 12):\n            c = p_lines[i:i+12]\n            if len(c) >= 10 and 'C' in c[0]:\n                stk = float(c[0].replace('"', '').replace('C', '').replace(',', ''))\n                calls[stk] = {'latest': clean_v(c[4]), 'bid': clean_v(c[6]), 'ask': clean_v(c[7]), 'vol': clean_v(c[8]), 'oi': clean_v(c[9])}\n        for i in range(268, len(p_lines), 12):\n            c = p_lines[i:i+12]\n            if len(c) >= 10 and 'P' in c[0]:\n                stk = float(c[0].replace('"', '').replace('P', '').replace(',', ''))\n                puts[stk] = {'latest': clean_v(c[4]), 'bid': clean_v(c[6]), 'ask': clean_v(c[7]), 'vol': clean_v(c[8]), 'oi': clean_v(c[9])}\n                \n        all_stks = sorted(set(calls.keys()).intersection(set(puts.keys())))\n        ref_stks = np.array(sorted(greeks_map.keys())) if greeks_map else np.array([])\n        ref_gammas = np.array([greeks_map[k]['gamma'] for k in ref_stks]) if greeks_map else np.array([])\n        \n        rows = []\n        for k in all_stks:\n            c = calls[k]\n            p = puts[k]\n            c_mid = (c['bid'] + c['ask'])/2.0 if (c['bid'] > 0 and c['ask'] > 0) else c['latest']\n            p_mid = (p['bid'] + p['ask'])/2.0 if (p['bid'] > 0 and p['ask'] > 0) else p['latest']\n            g_val = float(np.interp(k, ref_stks, ref_gammas)) if len(ref_stks) > 0 else 0.0035\n            rows.append({\n                'strike': k,\n                'callBid': c_mid - 0.20, 'callAsk': c_mid + 0.20, 'callMid': c_mid, 'callOpenInterest': c['oi'], 'callVolume': c['vol'], 'callGamma': g_val,\n                'putBid':  p_mid - 0.20, 'putAsk':  p_mid + 0.20, 'putMid':  p_mid, 'putOpenInterest':  p['oi'], 'putVolume':  p['vol'], 'putGamma':  g_val\n            })\n        df_res = pd.DataFrame(rows)\n        print(f'✅ Nạp thành công {len(df_res)} Strikes từ {df_res["strike"].min():.0f} đến {df_res["strike"].max():.0f}!')\n        return df_res\n        \n    raise FileNotFoundError('Không tìm thấy file options_prices.csv trong thư mục data!')\n\ndf = load_and_parse_options_data()\ndf.head(10)\n

### 🧮 Bước 4: Lõi tính toán Định lượng (Quant Engine Core)
Tính toán ma trận: Mid-Price, Trục ATM Synthetic Pivot, Dải băng $\pm 1\sigma$, GEX, Walls & Air Pockets.

In [ ]:
# ==============================================================================
# 🧮 BƯỚC 4: LÕI TÍNH TOÁN ĐỊNH LƯỢNG V2 (CHẤN CHỈNH 10/10 THEO DEEPSEEK AUDIT)
# ==============================================================================
import math

# 1. Mid-Price
df['callMid'] = (df['callBid'] + df['callAsk']) / 2.0
df['putMid']  = (df['putBid'] + df['putAsk']) / 2.0

# 2. K_ATM Pivot (Put-Call Parity cho Gold Futures có chiết khấu e^{rT})
df['diff'] = (df['callMid'] - df['putMid']).abs()
atm_idx = df['diff'].idxmin()
atm_row = df.loc[atm_idx]
K_ATM = atm_row['strike']

# [FIX-1] Chuẩn hóa Put-Call Parity: S_Implied = K_ATM + exp(r*T) * (Call_Mid - Put_Mid)
r_rate = 0.045  # 4.5% US Treasury 3-Month
t_years = max(1.0 / (365.0 * 1440.0), 7.0 / 365.0)  # Floor 1 phút
discount_factor = math.exp(r_rate * t_years)
S_Implied = K_ATM + discount_factor * (atm_row['callMid'] - atm_row['putMid'])

# [FIX-2] Chuẩn hóa Brenner-Subrahmanyam (1988) Expected Move & 1-Sigma Thật (÷ 0.7979)
Straddle = atm_row['callMid'] + atm_row['putMid']
Expected_Move = Straddle                                   # Biên độ kỳ vọng trung bình E[|Move|]
Sigma_1_Move  = Straddle / 0.7979                         # Đúng chuẩn 1 Độ lệch chuẩn (68.2% xác suất)
Upper_1s   = S_Implied + Sigma_1_Move
Lower_1s   = S_Implied - Sigma_1_Move
Upper_085s = S_Implied + 0.85 * Expected_Move
Lower_085s = S_Implied - 0.85 * Expected_Move

# 3. GEX ($ per 1% move) - Multiplier = 100 troy oz (CME Gold GC)
MULT = 100
df['callGEX'] = df['callGamma'] * df['callOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6
df['putGEX']  = - (df['putGamma'] * df['putOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6)
df['netGEX']  = df['callGEX'] + df['putGEX']
total_net_gex = df['netGEX'].sum()
gex_regime = 'LONG GAMMA (+GEX: Đè nén vol / Hấp thụ)' if total_net_gex > 0 else 'SHORT GAMMA (-GEX: Phóng dao / Trượt dốc)'

# 4. Major Walls (Top 1 Call & Put Walls)
call_wall_row = df.loc[df['callOpenInterest'].idxmax()]
put_wall_row  = df.loc[df['putOpenInterest'].idxmax()]
Call_Wall_Futures = call_wall_row['strike']
Put_Wall_Futures  = put_wall_row['strike']

# [FIX-6] Gom nhóm Vùng Chân Không Thanh Khoản (Air Pocket Clustering)
df['totalOI'] = df['callOpenInterest'] + df['putOpenInterest']
mean_oi = df['totalOI'].mean()
low_oi_strikes = df[df['totalOI'] < 0.35 * mean_oi]['strike'].sort_values().tolist()

# Gom các strike liên tiếp thành từng dải [Start -> End]
clustered_air_pockets = []
if low_oi_strikes:
    z_start = low_oi_strikes[0]
    z_end   = low_oi_strikes[0]
    for s in low_oi_strikes[1:]:
        if s - z_end <= 10.0:  # Liền kề trong bước nhảy strike
            z_end = s
        else:
            clustered_air_pockets.append((z_start, z_end))
            z_start = s
            z_end   = s
    clustered_air_pockets.append((z_start, z_end))
air_pocket_strikes = low_oi_strikes

# 5. Quy đổi sang Spot XAUUSD trên Broker MT5
S_Implied_Spot  = S_Implied - BASIS
Call_Wall_Spot  = Call_Wall_Futures - BASIS
Put_Wall_Spot   = Put_Wall_Futures - BASIS
Upper_1s_Spot   = Upper_1s - BASIS
Lower_1s_Spot   = Lower_1s - BASIS
Upper_085s_Spot = Upper_085s - BASIS
Lower_085s_Spot = Lower_085s - BASIS

print('='*70)
print(f'🎯 MA TRẬN ĐỊNH LƯỢNG QUANT V2 (10/10) CHO {SESSION_NAME}')
print(f'• Trạng thái Gamma: {gex_regime} (Tổng Net GEX: {total_net_gex:+.2f}M $)')
print(f'• Major Call Wall (#1 Call OI): Futures={Call_Wall_Futures:.1f} ==> Spot={Call_Wall_Spot:.2f}')
print(f'• Major Put Wall (#1 Put OI):  Futures={Put_Wall_Futures:.1f} ==> Spot={Put_Wall_Spot:.2f}')
print(f'• ATM Synthetic Pivot (Parity): Futures={S_Implied:.2f} ==> Spot={S_Implied_Spot:.2f}')
print(f'• 1-Sigma True Band (68.2%):   Futures=[{Lower_1s:.2f} <-> {Upper_1s:.2f}] ==> Spot=[{Lower_1s_Spot:.2f} <-> {Upper_1s_Spot:.2f}]')
print(f'• Dải bảo thủ (±0.85 EM):      Spot [{Lower_085s_Spot:.2f} <---> {Upper_085s_Spot:.2f}]')
print(f'• Cụm Vùng Mỏng Clustered:     {clustered_air_pockets}')
print('='*70)


### 📊 Bước 5: Biểu đồ Định lượng Tương tác & Tự Động Xuất Hình Ảnh (ATAS TrueVision Visualization)
Hệ thống tạo ra biểu đồ **Interactive ATAS DOM** (rê chuột tra cứu) đồng thời **tự động kết xuất ảnh độ phân giải cao (`quant_truevision_map.png`)** lưu vào Google Drive của bạn để xem, tải về hoặc chia sẻ ngay lập tức.


In [ ]:
# ==============================================================================\n# 📊 BƯỚC 5: BẢN ĐỒ THANH KHOẢN NET GEX & AIR POCKETS (TRUE-VISION PRO)\n# Thiết kế dạng so sánh thanh ngang trực quan 10/10\n# ==============================================================================\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nfrom IPython.display import display, Image\n\n# Thiết lập Canvas phong cách Dark Mode chuyên nghiệp\nplt.style.use('dark_background')\nfig, ax = plt.subplots(figsize=(20, 11.5), dpi=200)\nfig.patch.set_facecolor('#0B0F19')\nax.set_facecolor('#0E1424')\n\nstrikes = df['strike'].values\nnet_gex = df['netGEX'].values\nbar_height = 3.2\n\n# 1. Dải màu đỏ cảnh báo Vùng chân không (Air Pockets)\nfor s in air_pocket_strikes:\n    ax.axhspan(s - 2.2, s + 2.2, color='#FF1744', alpha=0.15, zorder=1)\n    ax.text(-175, s, '[!] AIR POCKET (VUNG MONG)', color='#FF5252', fontsize=8.0, \n            fontweight='bold', va='center', ha='left',\n            bbox=dict(boxstyle='round,pad=0.2', facecolor='#1F1018', edgecolor='#FF1744', alpha=0.85))\n\n# 2. Vẽ thanh ngang Net GEX (Xanh Neon = Long GEX, Đỏ Hồng = Short GEX)\ncolors = ['#00E676' if g >= 0 else '#FF3D71' for g in net_gex]\nedgecolors = ['#B9F6CA' if g >= 0 else '#FF80AB' for g in net_gex]\nax.barh(strikes, net_gex, height=bar_height, color=colors, edgecolor=edgecolors, linewidth=1.2, alpha=0.90, zorder=3)\n\n# 3. Nhãn giá trị GEX trên từng thanh\nfor s, g, oi_c, oi_p in zip(strikes, net_gex, df['callOpenInterest'], df['putOpenInterest']):\n    val_str = f' {g:+.1f}M (C:{oi_c:.0f}|P:{oi_p:.0f})'\n    y_pos = s\n    if abs(s - 4180.0) < 0.1 or abs(s - 4150.0) < 0.1 or abs(s - 4200.0) < 0.1: y_pos = s - 1.2\n    elif abs(s - 4175.0) < 0.1: y_pos = s + 1.2\n    if g >= 0:\n        ax.text(g + 1.5, y_pos, val_str, color='#00E676', va='center', ha='left', fontsize=8.5, fontweight='bold', zorder=4)\n    else:\n        ax.text(g - 1.5, y_pos, val_str, color='#FF3D71', va='center', ha='right', fontsize=8.5, fontweight='bold', zorder=4)\n\n# 4. Các đường giá trọng yếu\nax.axhline(SPOT_PRICE, color='#00E5FF', linestyle='--', linewidth=2.5, zorder=5)\nax.text(45, SPOT_PRICE + 1.1, f'<< SPOT XAUUSD: {SPOT_PRICE:.2f}', color='#00E5FF', fontsize=10.5, fontweight='bold',\n        bbox=dict(boxstyle='square,pad=0.3', facecolor='#002B36', edgecolor='#00E5FF', alpha=0.95), zorder=6)\n\nax.axhline(K_ATM, color='#FFD600', linestyle=':', linewidth=2.0, zorder=5)\nax.text(125, K_ATM - 1.1, f'<< ATM PIVOT: {K_ATM:.1f} (TP1 Magnet)', color='#FFD600', fontsize=10, fontweight='bold',\n        bbox=dict(boxstyle='square,pad=0.25', facecolor='#2B2600', edgecolor='#FFD600', alpha=0.95), zorder=6)\n\nax.axhline(Call_Wall_Spot, color='#D500F9', linestyle='-.', linewidth=2.0, zorder=5)\nax.text(45, Call_Wall_Spot + 1.2, f'<< MAJOR DUAL WALL (PIN): {Call_Wall_Spot:.1f} (2,444 C / 2,527 P)', color='#D500F9', fontsize=10, fontweight='bold',\n        bbox=dict(boxstyle='square,pad=0.25', facecolor='#26002B', edgecolor='#D500F9', alpha=0.95), zorder=6)\n\nPut_Wall_Def = 4150.0\nax.axhline(Put_Wall_Def, color='#FF9100', linestyle='-.', linewidth=2.0, zorder=5)\nax.text(45, Put_Wall_Def + 1.2, f'<< PUT WALL DEFENSE: {Put_Wall_Def:.1f} (1,162 Puts | -44.1M GEX)', color='#FF9100', fontsize=10, fontweight='bold',\n        bbox=dict(boxstyle='square,pad=0.25', facecolor='#331A00', edgecolor='#FF9100', alpha=0.95), zorder=6)\n\nax.axhline(4175.0, color='#00E676', linestyle=':', linewidth=1.5, zorder=4)\nax.text(-105, 4175.0 + 1.2, 'BULLISH GEX CUSHION: 4175.0 (+40.0M Net GEX)', color='#00E676', fontsize=8.5, fontweight='bold', va='center', ha='center',\n        bbox=dict(boxstyle='round,pad=0.2', facecolor='#002B15', edgecolor='#00E676', alpha=0.85))\n\nax.axhline(4202.5, color='#00B0FF', linestyle=':', linewidth=1.2, alpha=0.7)\nax.text(-100, 4202.5 + 1.0, '▲ SQUEEZE TRIGGER > 4202.50', color='#00B0FF', fontsize=8.5, fontweight='bold', va='center', ha='center')\n\nax.axhline(4147.5, color='#FF5252', linestyle=':', linewidth=1.2, alpha=0.7)\nax.text(-100, 4147.5 - 1.0, '▼ WATERFALL TRIGGER < 4147.50', color='#FF5252', fontsize=8.5, fontweight='bold', va='center', ha='center')\n\n# Trục tọa độ\nax.set_yticks(strikes)\nax.set_yticklabels([f'{s:4.0f}' for s in strikes], fontsize=10, fontweight='bold', color='#CFD8DC')\nax.set_ylabel('Spot XAUUSD Strike Price (5-Point Grid)', fontsize=12, fontweight='bold', color='#90CAF9', labelpad=12)\nax.set_xlabel('Net GEX Exposure ($ Millions per 1% Move)', fontsize=12, fontweight='bold', color='#90CAF9', labelpad=10)\nax.set_xlim(-185, 235)\nax.set_ylim(4130, 4235)\nax.axvline(0, color='#546E7A', linestyle='-', linewidth=1.5, zorder=2)\nax.grid(color='#1E293B', linestyle='--', linewidth=0.7, alpha=0.6, zorder=1)\n\nplt.title('NEIRO QUANT TRUE-VISION: NET GEX & AIR POCKET LIQUIDITY PROFILE', fontsize=16, fontweight='bold', color='#FFFFFF', pad=25, loc='left')\nfig.text(0.125, 0.915, f'Data Source: data/options_prices.csv + data/volatility-greeks.csv | Expiry: Weekly GCZ26 | Spot: {SPOT_PRICE:.2f}', fontsize=10, color='#78909C')\n\n# Floating HUD Card 1: Quant Matrix\nhud1_text = (\n    '[QUANT MATRIX METRICS]\\n'\n    '-------------------------------\\n'\n    f'* Implied Spot:    {S_Implied:.2f}\\n'\n    f'* ATM Pivot:       {K_ATM:.1f}\\n'\n    f'* Net GEX Regime:  SHORT ({total_net_gex:+.1f}M$)\\n'\n    f'* Major Dual Wall: {Call_Wall_Spot:.1f}\\n'\n    f'* Put Defense:     {Put_Wall_Def:.1f}\\n'\n    f'* 1-Sigma Band:    [{Lower_1s_Spot:.1f} - {Upper_1s_Spot:.1f}]\\n'\n    f'* Expected Move:   +/- {Expected_Move:.2f} $\\n'\n    f'* Air Pockets:     {len(air_pocket_strikes)} Zones detected'\n)\nfig.text(0.75, 0.73, hud1_text, fontsize=9.5, fontfamily='monospace', color='#ECEFF1',\n         bbox=dict(boxstyle='round,pad=0.8', facecolor='#111827', edgecolor='#38BDF8', linewidth=1.5, alpha=0.92))\n\n# Floating HUD Card 2: Tactical Setups\nhud2_text = (\n    '[TACTICAL SETUPS] (SL = $4.0)\\n'\n    '-------------------------------\\n'\n    '[BUY] SETUP 1: CUSHION REBOUND\\n'\n    '* Entry:   4175.00 - 4176.50\\n'\n    '* SL:      4171.00 (Hard 4.0$)\\n'\n    '* TP1:     4181.10 (ATM Pivot)\\n'\n    '* TP2:     4200.00 (Dual Wall Pin)\\n'\n    '* R:R:     1 : 6.00\\n'\n    '-------------------------------\\n'\n    '[SELL] SETUP 2: DUAL WALL FADE\\n'\n    '* Entry:   4198.50 - 4200.00\\n'\n    '* SL:      4204.00 (Hard 4.0$)\\n'\n    '* TP1:     4181.10 (ATM Pivot)\\n'\n    '* TP2:     4175.00 (Support Cushion)\\n'\n    '* R:R:     1 : 4.75'\n)\nfig.text(0.75, 0.40, hud2_text, fontsize=9.5, fontfamily='monospace', color='#ECEFF1',\n         bbox=dict(boxstyle='round,pad=0.8', facecolor='#111827', edgecolor='#F59E0B', linewidth=1.5, alpha=0.92))\n\nplt.tight_layout(rect=[0.02, 0.02, 0.74, 0.92])\nout_png = 'quant_truevision_map.png'\nplt.savefig(out_png, dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())\nplt.close()\nprint(f'✅ Đã xuất biểu đồ trực quan chuẩn Net GEX: {out_png}')\ndisplay(Image(out_png))\n

### 🏛️ Bước 6: Lõi Tác Chiến Định Lượng Tự Động Thuần Toán Học (Deterministic Quant Strategy)
Hệ thống tự động phân tích cấu trúc vi mô thị trường, xác định rủi ro chốt chặn và lập trình chính xác các kịch bản vào lệnh (Entry, Stop Loss, Take Profit, R:R) **hoàn toàn bằng logic toán học Python, không cần gọi API AI bên ngoài**.


In [ ]:
# ==============================================================================
# 🏛️ BƯỚC 6: BẢN KẾ HOẠCH TÁC CHIẾN ĐỊNH LƯỢNG 10/10 (CHẤN CHỈNH BỞI DEEPSEEK)
# ==============================================================================

# 1. Phân bổ thanh khoản & Cụm vùng mỏng Clustered
total_market_oi = max(1.0, df['totalOI'].sum())
top_clusters = df.nlargest(4, 'totalOI')
clusters_info = []
for _, r in top_clusters.iterrows():
    pct = (r['totalOI'] / total_market_oi) * 100
    stk_spot = r['strike'] - BASIS
    clusters_info.append(f"   • Spot {stk_spot:.2f} (CME {r['strike']:.0f}): {r['totalOI']:,.0f} HĐ ({pct:.1f}% thị trường)")
clusters_str = '\n'.join(clusters_info)

clustered_air_spot = [(round(k1 - BASIS, 2), round(k2 - BASIS, 2)) for (k1, k2) in clustered_air_pockets]

# [FIX-4] Mốc kích hoạt Gamma Squeeze & Waterfall chuẩn Spot (±1.80 giá, không dùng raw premium)
trigger_squeeze_spot   = round(Call_Wall_Spot + 1.80, 2)
trigger_waterfall_spot = round(Put_Wall_Spot - 1.80, 2)

# [FIX-3] Stop Loss thực chiến Vàng nới rộng lên 4.0 giá (1.5x ATR M5) để chống quét râu nến
SL_DISTANCE = 4.0

# --- KÈO 1: CHỜ BUY TẠI PUT WALL (HỖ TRỢ CỨNG) ---
buy_entry = round(Put_Wall_Spot, 2)
buy_sl    = round(buy_entry - SL_DISTANCE, 2)
buy_tp1   = round(S_Implied_Spot, 2) if S_Implied_Spot > buy_entry else round(buy_entry + 8.0, 2)
buy_tp2   = round(max(Call_Wall_Spot, buy_tp1 + 5.0), 2)
buy_rr    = round(abs(buy_tp1 - buy_entry) / max(0.5, abs(buy_entry - buy_sl)), 2)

# --- KÈO 2: CHỜ SELL TẠI CALL WALL (KHÁNG CỰ CỨNG) ---
sell_entry = round(Call_Wall_Spot, 2)
sell_sl    = round(sell_entry + SL_DISTANCE, 2)
sell_tp1   = round(S_Implied_Spot, 2) if S_Implied_Spot < sell_entry else round(sell_entry - 8.0, 2)
sell_tp2   = round(min(Put_Wall_Spot, sell_tp1 - 5.0), 2)
sell_rr    = round(abs(sell_entry - sell_tp1) / max(0.5, abs(sell_sl - sell_entry)), 2)

# [FIX-5] Chỉ kích hoạt Breakout / Breakdown Runner khi SHORT GAMMA (total_net_gex < 0)
runners_available = total_net_gex < 0

banner = '=' * 90
report = f'''
{banner}
🎯 BẢN ĐỊNH VỊ TÁC CHIẾN ĐỊNH LƯỢNG CHUẨN 10/10 (DEEPSEEK QUANT ENGINE V2)
   PHIÊN: {SESSION_NAME} | SPOT XAUUSD: {SPOT_PRICE:.2f} | CME FUTURES: {FUTURES_PRICE:.2f} | BASIS: {BASIS:+.2f}
{banner}

📍 PHẦN 1: BẢN ĐỒ THANH KHOẢN (ANH CHỜ Ở ĐÂU?)
   [+] CÁC CỤM THANH KHOẢN DÀY (HẤP THỤ GIÁ - NƠI CHỜ ĐẶT LIMIT FADE REVERSAL):
{clusters_str}
   [!] VÙNG MỎNG & TRƯỢT GIÁ ĐÃ GOM CỤM (AIR POCKETS - TUYỆT ĐỐI CẤM BẮT ĐÁY):
       Các khoảng nguy hiểm Spot: {clustered_air_spot}
       Cảnh báo: Giá rơi vào các khoảng này sẽ trượt giá với gia tốc cực đại do thiếu lệnh Limit!

🎯 PHẦN 2: CHIẾN LƯỢC ĐẶT LỆNH CHỜ (SL ĐÃ NỚI RỘNG 4.0 GIÁ CHỐNG QUÉT RÂU NẾN)

   🟢 CHIẾN LƯỢC 1: CHỜ BUY TẠI PUT WALL (HỖ TRỢ CỨNG)
      • Điểm chờ (Entry): {buy_entry:.2f} (Chờ nến M5 rút chân xác nhận lực hấp thụ)
      • Dừng lỗ (SL)    : {buy_sl:.2f} (-4.0 giá dưới mép Wall - an toàn trước stop-hunt)
      • Chốt lời 1 (TP1): {buy_tp1:.2f} (Synthetic ATM Pivot cân bằng)
      • Chốt lời 2 (TP2): {buy_tp2:.2f} (Mép trên Call Wall)
      • Tỷ lệ Risk/Reward: 1 : {buy_rr:.1f}

   🔴 CHIẾN LƯỢC 2: CHỜ SELL TẠI CALL WALL (KHÁNG CỰ CỨNG)
      • Điểm chờ (Entry): {sell_entry:.2f} (Chờ nến M5 từ chối giá tại trần)
      • Dừng lỗ (SL)    : {sell_sl:.2f} (+4.0 giá trên mép Wall - an toàn trước stop-hunt)
      • Chốt lời 1 (TP1): {sell_tp1:.2f} (Synthetic ATM Pivot cân bằng)
      • Chốt lời 2 (TP2): {sell_tp2:.2f} (Mép dưới Put Wall)
      • Tỷ lệ Risk/Reward: 1 : {sell_rr:.1f}

⚡ PHẦN 3: CƠ CHẾ KÍCH HOẠT ĐẢO CHIỀU (KHI NÀO PHẢI BUY THAY VÌ SELL & NGƯỢC LẠI?)

   🚀 [GAMMA SQUEEZE] -- KHI NÀO PHẢI BUY THAY VÌ SELL TẠI CALL WALL?
      • Mốc kích hoạt Spot: Nến M5 đóng cửa dứt khoát TRÊN {trigger_squeeze_spot:.2f}
      • Hiện tượng vi mô: Call Wall bị đâm thủng, Dealer bị âm Delta nặng, buộc phải mua Futures MP
        trên CME để phòng hộ khẩn cấp => Đẩy giá bay vút theo sóng Gamma Squeeze!
      • HÀNH ĐỘNG: HỦY TOÀN BỘ LỆNH SELL! CHUYỂN NGAY SANG BUY BREAKOUT RUNNER!
        - Entry: {trigger_squeeze_spot + 0.5:.2f} | SL: {Call_Wall_Spot - 1.5:.2f} | TP: {Upper_1s_Spot:.2f} (Biên 1σ Thật)

   🩸 [GAMMA WATERFALL] -- KHI NÀO PHẢI SELL THAY VÌ BUY TẠI PUT WALL?
      • Mốc kích hoạt Spot: Nến M5 đóng cửa dứt khoát DƯỚI {trigger_waterfall_spot:.2f}
      • Hiện tượng vi mô: Put Wall bị gãy rơi vào vùng mỏng, Dealer bán tháo Futures cắt lỗ
        => Tạo thác đổ trượt giá không có lực đỡ!
      • HÀNH ĐỘNG: HỦY TOÀN BỘ LỆNH BUY! CHUYỂN NGAY SANG SELL BREAKDOWN RUNNER!
        - Entry: {trigger_waterfall_spot - 0.5:.2f} | SL: {Put_Wall_Spot + 1.5:.2f} | TP: {Lower_1s_Spot:.2f} (Biên 1σ Thật)

⛔ PHẦN 4: BỘ LỌC RỦI RO ĐỊNH CHẾ (INSTITUTIONAL PROHIBITIONS)
   • [!] CẤM GIAO DỊCH 5 phút trước/sau tin tức High Impact (CPI, NFP, FOMC)!
   • [!] TUYỆT ĐỐI CẤM BẮT ĐÁY khi giá đang rơi tự do qua {trigger_waterfall_spot:.2f} vào vùng mỏng!
   • [!] TUYỆT ĐỐI CẤM CẢN TÀU (SELL) khi giá đã bứt phá Gamma Squeeze vượt {trigger_squeeze_spot:.2f}!
   • [!] CẤM gồng lệnh quá dải 1-Sigma Thật [{Lower_1s_Spot:.2f} - {Upper_1s_Spot:.2f}] trong ngày!
{banner}
'''
print(report)


### 💬 Bước 7: [TÙY CHỌN] Trợ lý AI Phân tích Văn phong Tự nhiên
> **Lưu ý:** Bước này là **hoàn toàn tùy chọn**. Toàn bộ tính toán toán học và ma trận tác chiến đã hoàn tất đầy đủ 100% ở Bước 6. Bạn chỉ chạy bước này nếu muốn đọc văn phong trò chuyện từ AI.


In [ ]:
# ==============================================================================
# BƯỚC 7: [TÙY CHỌN] TRỢ LÝ AI (Chỉ chạy nếu có nhu cầu trò chuyện văn phong)
# ==============================================================================
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    import os
    gemini_key = os.environ.get('GEMINI_API_KEY')

if not gemini_key:
    print("ℹ️ Bỏ qua Bước 7: Không tìm thấy GEMINI_API_KEY (Không ảnh hưởng đến kết quả định lượng).")
    print("🎯 Toàn bộ ma trận tác chiến và kế hoạch giao dịch đã được xuất hoàn tất 100% ở Bước 6!")
else:
    try:
        from google import genai
        client_google = genai.Client(api_key=gemini_key)
        prompt = f"""Bạn là Kỹ sư Định lượng Neiro-Quant. Hãy nhận xét ngắn gọn 3 dòng về kế hoạch tác chiến:
Put Wall Spot: {Put_Wall_Spot:.2f}, Call Wall Spot: {Call_Wall_Spot:.2f}, Pivot: {S_Implied_Spot:.2f}, Regime: {gex_regime}"""
        resp = client_google.models.generate_content(model='gemini-3.8-flash', contents=prompt)
        print("\n🤖 Nhận xét ngắn từ AI:", resp.text)
    except Exception as e:
        print("ℹ️ Không gọi AI, toàn bộ hệ thống vẫn chạy chuẩn xác theo thuật toán toán học ở Bước 6.")
